In [1]:
import pandas as pd
import yfinance as yf
import os
import re
import numpy as np

Create Directory if needed

In [2]:
os.makedirs(os.path.dirname("./saveData"), exist_ok=True)

Download Twitter Data

In [3]:
sentiment_path = "./savedData/datasets/gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged/versions/1/mbsa.csv"

if os.path.exists(sentiment_path):
    df_sentiment = pd.read_csv(sentiment_path)
    print("Sentiment data loaded")
else:
    os.environ["KAGGLEHUB_CACHE"] = "./savedData"
    import kagglehub

    # Download the Bitcoin tweet dataset
    path = kagglehub.dataset_download("gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged")
    print("Path to dataset files:", path)
    df_sentiment = pd.read_csv(sentiment_path)
    print("Sentiment data downloaded, saved and loaded")

Sentiment data loaded


Data Pipeline

In [4]:
# Delete Duplicates
df_sentiment = df_sentiment.drop_duplicates()

# Clean Dates
df_sentiment = df_sentiment[pd.to_datetime(df_sentiment['Date'], errors='coerce').notnull()]
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], errors='coerce')

# Clean and map Sentiment
sentiment_map = {'Positive': 1, 'Negative': -1}
df_sentiment = df_sentiment.dropna(subset=['Sentiment'])
df_sentiment = df_sentiment[df_sentiment['Sentiment'].isin(['Positive', 'Negative'])].copy()
df_sentiment['sentiment_score'] = df_sentiment['Sentiment'].map(sentiment_map)

# Clean Text
df_sentiment = df_sentiment.dropna(subset=['text'])

Optional: Remove emojis and url´s from tweets

In [5]:
# df_sentiment.loc[:, 'text'] = df_sentiment['text'].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+", '', x))
# df_sentiment.loc[:, 'text'] = df_sentiment['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [6]:
start_date = df_sentiment['Date'].min()
end_date = df_sentiment['Date'].max()
print("Minimales Datum: ", start_date, "\nMaximales Datum: ", end_date)

Minimales Datum:  2014-09-18 00:00:00 
Maximales Datum:  2019-11-23 00:00:00


Download Bitcoin Data

In [7]:
bitcoin_path = "./savedData/bitcoinPrice.csv"

if(os.path.exists(bitcoin_path)):
    df_bitcoin = pd.read_csv(bitcoin_path, header=0)
    df_bitcoin = df_bitcoin.dropna(subset=["Date"])
    df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])
    df_bitcoin.columns = ['Date', 'Close']
    print("Bitcoin data loaded")
else:
    ticker = "BTC-USD"
    df_bitcoin = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
    df_bitcoin.reset_index(inplace=True)
    df_bitcoin = df_bitcoin[['Date', 'Close']]
    df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])
    df_bitcoin.columns = ['Date', 'Close']
    df_bitcoin.to_csv(bitcoin_path, index=False)
    print("Bitcoin data downloaded, loaded and saved")

Bitcoin data loaded


In [8]:
# Calculate sentiment per day
df_daily_sentiment = df_sentiment.groupby('Date')['sentiment_score'].mean().reset_index()
df_daily_sentiment.rename(columns={'sentiment_score': 'avg_sentiment'}, inplace=True)
df_daily_sentiment['Date'] = pd.to_datetime(df_daily_sentiment['Date'])

Complete Sentiment DF with next day price and avgSentiment

In [9]:
df_sentiment = df_sentiment.merge(df_daily_sentiment, on='Date', how='left')
df_bitcoin['next_day_price'] = df_bitcoin['Close'].shift(-1)
df_sentiment = df_sentiment.merge(df_bitcoin[['Date', 'next_day_price']], on='Date', how='left')

In [ ]:
df_sentiment_path = "./savedData/df_sentiment.csv"

if os.path.exists(df_sentiment_path):
    df_sentiment = pd.read_csv(df_sentiment_path)
    print("Already Saved")
else:
    df_sentiment.to_csv(df_sentiment_path, index=False)
    print("Saved")

In [ ]:
df_sentiment.head()